# Notebook 13 — Hyperfine Coherence Specifies Control Fidelity

**Engineering statement:** Hyperfine coherence specifies the control budget available for transporting, addressing, entangling, and measuring neutral atoms.

Notebook 07 showed that array scale becomes useful through control. Notebook 13 starts from the resource that makes that control possible: long-lived hyperfine coherence.

Generated figures:

```text
13_hero_statement.png
13_control_budget.png
13_fidelity_vs_operations.png
13_budget_pipeline.png
13_time_allocation.png
13_design_surface.png
13_notebook_summary.png
```


In [ ]:
from pathlib import Path
import json
import zipfile
import math

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

NOTEBOOK_NAME = "13_hyperfine_coherence"
CWD = Path.cwd().resolve()

if (CWD / "notebooks").exists() or (CWD / "figures").exists():
    REPO_ROOT = CWD
elif CWD.name == "notebooks":
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / NOTEBOOK_NAME
RESULT_FIGURES = RESULTS / "figures"

for path in [NOTEBOOKS, FIGURES, RESULTS, RESULT_FIGURES]:
    path.mkdir(parents=True, exist_ok=True)

def save_png(fig, name, dpi=220):
    repo_path = FIGURES / f"{name}.png"
    result_path = RESULT_FIGURES / f"{name}.png"
    fig.savefig(repo_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(result_path, dpi=dpi, bbox_inches="tight")
    print(f"✓ Saved {repo_path}")
    print(f"✓ Saved {result_path}")
    return repo_path

print(f"Repo root: {REPO_ROOT}")
print(f"Figures:   {FIGURES}")
print(f"Results:   {RESULTS}")


## 1. Variables and abstractions

\[
N_{\mathrm{useful}}
\approx
\frac{T_{\mathrm{coh}}}{t_{\mathrm{operation}}}
\]

\[
F_{\mathrm{sequence}}
\approx
F_{\mathrm{op}}^{N_{\mathrm{operations}}}
\]


In [ ]:
variables = pd.DataFrame([
    {"symbol": "T_coh", "meaning": "available hyperfine coherence time", "role": "engineering budget"},
    {"symbol": "t_operation", "meaning": "characteristic control-operation time", "role": "budget cost"},
    {"symbol": "N_useful", "meaning": "useful operation count", "role": "available control depth"},
    {"symbol": "F_op", "meaning": "per-operation fidelity", "role": "local control quality"},
    {"symbol": "F_sequence", "meaning": "sequence fidelity", "role": "end-to-end reliability"},
    {"symbol": "T_margin", "meaning": "unused coherence margin", "role": "fault-tolerance headroom"},
])
variables_path = RESULTS / "coherence_variables.csv"
variables.to_csv(variables_path, index=False)
variables


## 2. Hero statement


In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 7.2))
ax.axis("off")

ax.text(0.5, 0.78, "Hyperfine Coherence Specifies\nControl Fidelity",
        ha="center", va="center", fontsize=33, weight="bold")
ax.text(0.5, 0.56,
        "Long-lived hyperfine states become an engineering budget\nfor transport, addressing, gates, and measurement.",
        ha="center", va="center", fontsize=17)
ax.text(0.5, 0.36, r"$T_{\mathrm{coh}} \rightarrow N_{\mathrm{useful}}$",
        ha="center", va="center", fontsize=26,
        bbox=dict(boxstyle="round,pad=0.65", linewidth=1.2, facecolor="white"))
ax.text(0.5, 0.18,
        "longer coherence  •  more operations  •  higher control fidelity  •  larger margin",
        ha="center", va="center", fontsize=14)

fig.tight_layout()
hero_fig = save_png(fig, "13_hero_statement")
plt.show()
hero_fig


## 3. Control budget


In [ ]:
budget_parts = pd.DataFrame([
    {"stage": "transport", "fraction": 0.14},
    {"stage": "addressing", "fraction": 0.08},
    {"stage": "entangling", "fraction": 0.24},
    {"stage": "measurement", "fraction": 0.10},
    {"stage": "remaining margin", "fraction": 0.44},
])
budget_parts["percent"] = 100 * budget_parts["fraction"]
budget_path = RESULTS / "control_budget.csv"
budget_parts.to_csv(budget_path, index=False)

fig, ax = plt.subplots(figsize=(11.5, 5.4))
ax.axis("off")

ax.text(0.5, 0.92, "Hyperfine coherence becomes a control budget", ha="center", va="center", fontsize=19, weight="bold")
ax.text(0.5, 0.84, "Transport, addressing, gates, and measurement spend coherent time.", ha="center", va="center", fontsize=13)

left = 0.12
bar_width = 0.76
y = 0.58
height = 0.10
x = left

for _, row in budget_parts.iterrows():
    w = bar_width * row["fraction"]
    rect = plt.Rectangle((x, y), w, height, fill=False, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + height/2, row["stage"], ha="center", va="center", fontsize=10)
    ax.text(x + w/2, y - 0.08, f'{row["percent"]:.0f}%', ha="center", va="center", fontsize=10)
    x += w

ax.text(left, y + 0.18, "available coherence window", ha="left", va="center", fontsize=12, weight="bold")
ax.text(0.5, 0.20, "Control fidelity depends on preserving enough remaining margin.", ha="center", va="center", fontsize=13)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
fig.tight_layout()

budget_fig = save_png(fig, "13_control_budget")
plt.show()
budget_parts


## 4. Fidelity versus operation count


In [ ]:
N = np.arange(0, 1001, 20)
fidelities = [0.9999, 0.9995, 0.9990, 0.9980]

fidelity_rows = []
for fop in fidelities:
    for n in N:
        fidelity_rows.append({
            "N_operations": int(n),
            "F_op": fop,
            "F_sequence": fop ** n,
        })

fidelity_df = pd.DataFrame(fidelity_rows)
fidelity_path = RESULTS / "fidelity_vs_operations.csv"
fidelity_df.to_csv(fidelity_path, index=False)

fig, ax = plt.subplots(figsize=(9.2, 5.4))
for fop in fidelities:
    sub = fidelity_df[fidelity_df["F_op"] == fop]
    ax.plot(sub["N_operations"], sub["F_sequence"], label=f"Fop = {fop}")

ax.set_xlabel("Operation count")
ax.set_ylabel("Sequence fidelity")
ax.set_title("Control fidelity decreases as operation count grows")
ax.set_ylim(0.0, 1.02)
ax.legend(title="Per-operation")
fig.tight_layout()

fidelity_fig = save_png(fig, "13_fidelity_vs_operations")
plt.show()
fidelity_df.head()


## 5. Budget pipeline


In [ ]:
pipeline = [
    ("Hyperfine\ncoherence", "state memory"),
    ("Transport\nfidelity", "move without losing state"),
    ("Addressing\nfidelity", "select without crosstalk"),
    ("Gate\nfidelity", "entangle reliably"),
    ("Logical\noperations", "structured computation"),
    ("Fault\ntolerance", "error-corrected pathway"),
]

fig, ax = plt.subplots(figsize=(10.5, 7.2))
ax.axis("off")

ax.text(0.5, 0.95, "Hyperfine coherence supports the control stack", ha="center", va="center", fontsize=20, weight="bold")
ax.text(0.5, 0.88, "Control fidelity is built by preserving state information through each operation layer.", ha="center", va="center", fontsize=12.5)

x = 0.5
y0 = 0.76
dy = 0.125

for i, (title, subtitle) in enumerate(pipeline):
    y = y0 - i * dy
    ax.text(x, y, f"{title}\n{subtitle}", ha="center", va="center", fontsize=12.5,
            bbox=dict(boxstyle="round,pad=0.55", linewidth=1.2, facecolor="white"))
    if i < len(pipeline) - 1:
        ax.annotate("", xy=(x, y - 0.075), xytext=(x, y - 0.045),
                    arrowprops=dict(arrowstyle="->", linewidth=1.4))

ax.text(0.5, 0.08, "coherence  →  control fidelity  →  logical operations  →  fault tolerance", ha="center", va="center", fontsize=13)

fig.tight_layout()
pipeline_fig = save_png(fig, "13_budget_pipeline")
plt.show()
pipeline_fig


## 6. Time allocation


In [ ]:
allocation_scenarios = pd.DataFrame([
    {"scenario": "conservative", "transport": 0.10, "addressing": 0.06, "gates": 0.18, "measurement": 0.08, "margin": 0.58},
    {"scenario": "balanced", "transport": 0.15, "addressing": 0.08, "gates": 0.26, "measurement": 0.11, "margin": 0.40},
    {"scenario": "stressed", "transport": 0.22, "addressing": 0.12, "gates": 0.34, "measurement": 0.14, "margin": 0.18},
])
allocation_path = RESULTS / "time_allocation.csv"
allocation_scenarios.to_csv(allocation_path, index=False)

fig, ax = plt.subplots(figsize=(10.5, 5.4))

categories = ["transport", "addressing", "gates", "measurement", "margin"]
lefts = np.zeros(len(allocation_scenarios))

for cat in categories:
    vals = allocation_scenarios[cat].values
    ax.barh(allocation_scenarios["scenario"], vals, left=lefts, label=cat)
    lefts += vals

ax.set_xlabel("Fraction of coherence window")
ax.set_title("Control operations allocate the coherence window")
ax.set_xlim(0, 1)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.28), ncol=5)
fig.tight_layout()

allocation_fig = save_png(fig, "13_time_allocation")
plt.show()
allocation_scenarios


## 7. Design surface


In [ ]:
coherence_times = np.array([1, 2, 5, 10, 20, 50])
operation_times_ms = np.array([0.05, 0.1, 0.25, 0.5, 1.0])

surface_rows = []
for T in coherence_times:
    for t_ms in operation_times_ms:
        surface_rows.append({
            "T_coh_s": T,
            "t_operation_ms": t_ms,
            "N_useful": T / (t_ms / 1000.0),
        })

surface = pd.DataFrame(surface_rows)
surface_path = RESULTS / "design_surface.csv"
surface.to_csv(surface_path, index=False)

pivot = surface.pivot(index="t_operation_ms", columns="T_coh_s", values="N_useful")

fig, ax = plt.subplots(figsize=(9.5, 5.6))
im = ax.imshow(pivot.values, aspect="auto", origin="lower")

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c) for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([str(i) for i in pivot.index])

ax.set_xlabel("Coherence time (s)")
ax.set_ylabel("Operation time (ms)")
ax.set_title("Longer coherence expands feasible operation count")

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.values[i, j]:.0f}", ha="center", va="center", fontsize=9)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Useful operation count")
fig.tight_layout()

surface_fig = save_png(fig, "13_design_surface")
plt.show()
surface.head()


## 8. Notebook summary


In [ ]:
summary_steps = [
    ("Hyperfine\ncoherence", "engineering budget"),
    ("Reliable\ncontrol", "transport + gates"),
    ("Logical\nlayouts", "structured operations"),
    ("Fault-tolerant\ncomputation", "scalable pathway"),
]

fig, ax = plt.subplots(figsize=(12, 5.6))
ax.axis("off")

ax.text(0.5, 0.92, "Notebook 13 summary", ha="center", va="center", fontsize=19, weight="bold")
ax.text(0.5, 0.84, "Hyperfine coherence specifies control fidelity.", ha="center", va="center", fontsize=13.5)

for i, (title, subtitle) in enumerate(summary_steps):
    x = 0.16 + i * 0.23
    ax.text(x, 0.55, title, ha="center", va="center", fontsize=13, weight="bold",
            bbox=dict(boxstyle="round,pad=0.55", linewidth=1.2, facecolor="white"))
    ax.text(x, 0.33, subtitle, ha="center", va="center", fontsize=11)
    if i < len(summary_steps) - 1:
        ax.annotate("", xy=(x + 0.14, 0.55), xytext=(x + 0.07, 0.55),
                    arrowprops=dict(arrowstyle="->", linewidth=1.4))

ax.text(0.5, 0.12, "Notebook 17 asks how coherent transport preserves this fidelity during motion.", ha="center", va="center", fontsize=12.5)

fig.tight_layout()
summary_fig = save_png(fig, "13_notebook_summary")
plt.show()
summary_fig


## 9. Takeaways


In [ ]:
takeaways = pd.DataFrame([
    {"takeaway": "Hyperfine coherence defines an available control budget."},
    {"takeaway": "Every control operation spends part of that budget."},
    {"takeaway": "Sequence fidelity decreases as operation count increases."},
    {"takeaway": "Longer coherence expands feasible operation count."},
    {"takeaway": "Coherent transport becomes the next control-fidelity test."},
])
takeaways_path = RESULTS / "notebook_13_takeaways.csv"
takeaways.to_csv(takeaways_path, index=False)
takeaways


## 10. Save notebook manifest


In [ ]:
notebook_manifest = {
    "notebook": "13_hyperfine_coherence_specifies_control_fidelity.ipynb",
    "title": "Hyperfine Coherence Specifies Control Fidelity",
    "primary_specification": "hyperfine coherence specifies control fidelity",
    "statement": "Hyperfine coherence specifies the control budget available for transporting, addressing, entangling, and measuring neutral atoms.",
    "outputs": {
        "variables_csv": str(variables_path.relative_to(REPO_ROOT)),
        "budget_csv": str(budget_path.relative_to(REPO_ROOT)),
        "fidelity_csv": str(fidelity_path.relative_to(REPO_ROOT)),
        "allocation_csv": str(allocation_path.relative_to(REPO_ROOT)),
        "surface_csv": str(surface_path.relative_to(REPO_ROOT)),
        "takeaways_csv": str(takeaways_path.relative_to(REPO_ROOT)),
        "hero_figure": str(hero_fig.relative_to(REPO_ROOT)),
        "control_budget_figure": str(budget_fig.relative_to(REPO_ROOT)),
        "fidelity_figure": str(fidelity_fig.relative_to(REPO_ROOT)),
        "pipeline_figure": str(pipeline_fig.relative_to(REPO_ROOT)),
        "allocation_figure": str(allocation_fig.relative_to(REPO_ROOT)),
        "surface_figure": str(surface_fig.relative_to(REPO_ROOT)),
        "summary_figure": str(summary_fig.relative_to(REPO_ROOT)),
    },
}

manifest_path = RESULTS / "13_hyperfine_coherence_manifest.json"
manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding="utf-8")
notebook_manifest


## 11. Download artifacts


In [ ]:
from IPython.display import FileLink, display

zip_path = RESULTS / "13_hyperfine_coherence_artifacts.zip"
paths_to_include = [FIGURES, RESULTS]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)
        if item.is_dir():
            for path in item.rglob("*"):
                if not path.is_file():
                    continue
                if path.resolve() == zip_path.resolve():
                    continue
                try:
                    archive_name = path.relative_to(REPO_ROOT)
                except ValueError:
                    archive_name = path.name
                zf.write(path, archive_name)
        elif item.exists():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
